In [3]:
# Lab 3: Text Generation Using GPT-2

from pathlib import Path
import random
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

# ---------------- Step 1: Load the model ----------------
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

# ---------------- Step 2: Compare prompts and decoding ----------------
def continue_text(prompt, *, sample=False, temperature=0.8, seed=42):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    options = {"do_sample": sample, "max_new_tokens": 45,
               "pad_token_id": tokenizer.eos_token_id}
    if sample:
        options.update(temperature=temperature, top_p=0.9)
    with torch.no_grad():
        tokens = model.generate(**inputs, **options)
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):],
                            skip_special_tokens=True).strip()

prompts = [
    "A story about a robot begins like this:",
    "In a school library, a tiny robot found a lost book. It",
]
for prompt in prompts:
    print("\nPROMPT:", prompt)
    print("GREEDY:", continue_text(prompt))
    for temp in (0.6, 1.0):
        for seed in (42, 43):
            print(temp, seed, continue_text(prompt, sample=True,
                                            temperature=temp, seed=seed))

# ---------------- Step 3: Fine-tune on study tip examples ----------------
examples = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
]

tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=2e-5)

for epoch in range(3):
    tuned.train()
    for topic, answer in examples:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
    print("Epoch:", epoch + 1, "last example loss:", round(loss.item(), 3))

tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")

held_out = ["recursion", "matrices", "binary search"]
base_model = model
for topic in held_out:
    prompt = f"Topic: {topic}\nStudy tip:"
    model = base_model
    print(topic, "base:", continue_text(prompt))
    model = tuned.eval()
    print(topic, "tuned:", continue_text(prompt))
model = base_model

Device: cpu

PROMPT: A story about a robot begins like this:
GREEDY: A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed
0.6 42 The robot was a humanoid robot with a heart, eyes, and hands that looked like a human, but its skin was red. The robot had a small, muscular body and was very muscular. It was also very slow.
0.6 43 The robot, a robot named "The Robot," is trying to figure out what happened to its owner, who is a woman. The robot tries to figure out how to get back to her, and then it has to figure
1.0 42 The Robot Man was brought to Earth by the Galactic government, to escape the dark and deadly dangers of space. The robot is the most powerful of all the robots, and even better. It has a long memory for what was
1.0 43 The person in charge of the robots, his robot friend, or even their dog, is trying to be good. But as soo

In [2]:
%pip install -U "transformers>=4.51,<5" "huggingface-hub<1.0"
%pip install "numpy<2"

Note: you may need to restart the kernel to use updated packages.
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.2 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.2.0 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [2]:
%pip install -U ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.6
    Uninstalling widgetsnbextension-3.6.6:
      Successfully uninstalled widgetsnbextension-3.6.6
  Attempting uninstall: jupyterlab_widgets
    Found existing installation: jupyterlab-widgets 1.0.0
    Uninstalling jupyterlab-widgets-1.0.0:
      Successfully uninstalled jupyterlab-widgets-1.0.0
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.8.1
    Uninstalling ipywidgets-7.8.1:
      Successfully uninstalled ipywidgets-7.8.1
Note: you may need to restart the kernel to use updated packages.
